In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/mimiciv/3.1/ed'

DEMO

In [17]:
AGES = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]
AGE_LABELS = [f'Age/{AGES[i]}-{AGES[i+1]}' for i in range(0, len(AGES)-1)]
def age_str(x):
    ind = np.searchsorted(AGES, x, 'right')
    return AGE_LABELS[ind-1]

In [18]:
def race_str(x):
    if 'ASIAN' in x:
        return 'RACE/ASIAN'
    elif 'BLACK' in x:
        return 'RACE/BLACK'
    elif 'WHITE' in x:
        return 'RACE/WHITE'
    elif 'HISPANIC' in x:
        return 'RACE/HISPANIC'
    elif 'BLACK' in x:
        return 'RACE/BLACK'
    elif 'NATIVE' in x:
        return 'RACE/AMERICAN'
    elif x == 'PORTUGUESE': return 'RACE/PORTUGUESE'
    elif x == 'SOUTH AMERICAN': return 'RACE/SOUTH'
    else:
        return 'RACE/UNKNOWN'

In [28]:
stays_df = pd.read_csv(os.path.join(DATA_DIR, 'edstays.csv'))
stay_df = stays_df.drop(columns=['arrival_transport', 'disposition'])

In [29]:
stay_df['Race_str'] = stay_df['race'].apply(race_str)
stay_df['Gender_str'] = 'Gender/' + stay_df['gender']

In [30]:
stay_df['Gender_ind'] = pd.factorize(stay_df['gender'])[0]
stay_df['Race_ind'] = pd.factorize(stay_df['race'])[0]

In [32]:
stay_df.to_csv(os.path.join(DATA_DIR, 'demo.csv'), index=False)

STRING VALUES

In [3]:
stays_df = pd.read_csv(os.path.join(DATA_DIR, 'edstays.csv'))
vital_df = pd.read_csv(os.path.join(DATA_DIR, 'vitalsign.csv'))
triage_df = pd.read_csv(os.path.join(DATA_DIR, 'triage.csv'))

In [4]:
triage_df = triage_df.merge(stays_df[['stay_id', 'intime']], on='stay_id')
triage_df = triage_df.rename(columns={'intime': 'charttime'})
vital_df = pd.concat([triage_df, vital_df]).sort_values(['subject_id', 'charttime'])

In [ ]:
complaint_df = vital_df[['subject_id', 'stay_id', 'chiefcomplaint', 'charttime']]
complaint_df = complaint_df.dropna(subset=['chiefcomplaint'])

rhythm_df = vital_df[['subject_id', 'stay_id', 'rhythm', 'charttime']]
rhythm_df = rhythm_df.dropna(subset=['rhythm'])

In [ ]:
pain_df = vital_df[['subject_id', 'stay_id', 'pain', 'charttime']]
pain_df = pain_df.dropna(subset=['pain'])

In [20]:
vital_df = vital_df.drop(columns=['chiefcomplaint', 'rhythm', 'pain'])
df = None
for c in vital_df.columns:
    if c == 'stay_id' or c == 'charttime' or c == 'subject_id': continue
    df_c = vital_df[['subject_id', 'stay_id', 'charttime'] + [c]]
    df_c = df_c.rename(columns={c:'value'})
    df_c['event'] = c
    if df is None: df = df_c
    else: df = pd.concat([df, df_c])
df = df.dropna(subset=['value'])

In [21]:
import re

pattern = r'\b\d+[:/-]\d+(?:[:/-]\d+)*\b|\b\d+\b'

def extract_first_number(match_list):
    result = []
    for match in match_list:
        # Use regex to extract the leading digits before any non-digit char
        m = re.match(r'^(\d+)', match)
        if m:
            result.append(m.group(1))
        else:
            result.append(match)  # fallback, should rarely happen
    return result

In [22]:
pain_df = pain_df[~(pain_df['pain'].str.contains('sleep|unable|uta|u', case=False))]
pain_df['pain'] = pain_df['pain'].str.findall(pattern).apply(extract_first_number).apply(lambda nums: max(map(int, nums)) if nums else None)
pain_df = pain_df.dropna(subset=['pain'])

In [25]:
pain_df['event'] = 'pain'
pain_df = pain_df.rename(columns={'pain': 'value'})

In [32]:
df = pd.concat([df, pain_df], ignore_index=True)

In [34]:
df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)

BUCKETS

In [39]:
import pickle

In [ ]:
NUM_COLS = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain']
scale = 10_000

In [41]:
num_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))

In [ ]:
buckets = {}
buckets['acuity'] = [1, 2, 3, 4, 5]
for c in NUM_COLS:
    df_c = num_df[num_df['event'] == c]
    bucket = np.percentile(df_c['value'], np.arange(0, 110, 10), method='lower')
    bucket = bucket[np.insert(np.diff(bucket) != 0, 0, True)]
    bucket = np.array([
            np.floor(bucket[0] * scale) / scale,             # floor first
            *[round(b, ndigits=4) for b in bucket[1:-1]],         # round middle
            np.ceil(bucket[-1] * scale) / scale              # ceil last
            ])
    buckets[c] = bucket[np.concatenate(([True], bucket[1:] != bucket[:-1]))]

In [52]:
with open(os.path.join(DATA_DIR, 'ed_buckets.pkl'), 'wb') as f:
    pickle.dump(buckets, f)

EVENTVAL

In [53]:
def bucket_eventval(event, val, d):
    buckets = d[event]
    if type(buckets) == str: 
        return buckets
    ind = np.searchsorted(buckets, val, side='right')
    if ind == len(buckets):
        eventval = f"{event}|{buckets[ind-1]}-"
    else:
        eventval = f"{event}|{buckets[ind-1]}-{buckets[ind]}"
    return eventval

In [54]:
def bucket_ind(event, val, d):
    buckets = d[event]
    if type(buckets) == str: 
        return 0
    ind = np.searchsorted(buckets, val, side='right')
    return ind

In [ ]:
with open(os.path.join(DATA_DIR, 'ed_buckets.pkl'), 'rb') as file:
    buckets = pickle.load(file)

In [ ]:
num_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))

In [58]:
num_df['eventval'] = num_df.apply(lambda x: bucket_eventval(x['event'], x['value'], buckets), axis=1)
num_df['bucket'] = num_df.apply(lambda x: bucket_ind(x['event'], x['value'], buckets), axis=1)

In [ ]:
num_df = num_df.rename(columns={'value': 'Value', 'charttime': 'Time', 'event': 'Measure'})

In [61]:
num_df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)

MED

In [29]:
med_df = pd.read_csv(os.path.join(DATA_DIR, 'pyxis.csv'))

In [30]:
import networkx as nx

med_df['gsn'] = med_df['gsn'].astype(str)
med_df['name'] = med_df['name'].astype(str)

G = nx.Graph()
edges = med_df[['gsn', 'name']].drop_duplicates().values.tolist()
G.add_edges_from(edges)


components = list(nx.connected_components(G))
node_to_group = {node: i for i, comp in enumerate(components) for node in comp}

In [31]:
med_df['group'] = med_df.apply(
    lambda row: node_to_group.get(row['gsn']) or node_to_group.get(row['name']),
    axis=1
)

In [32]:
med_df = med_df[['subject_id', 'stay_id', 'charttime', 'group', 'name']]
med_df['eventval'] = 'MED/' + med_df['group'].astype(str)
med_df['bucket'] = 0

In [ ]:
med_df = med_df.rename(columns={'charttime': 'Time', 'group': 'Measure', 'stay_id'})

In [10]:
med_df.to_csv(os.path.join(DATA_DIR, 'med.csv'), index=False)

ARRIVAL TIME

In [ ]:
num_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))
med_df = pd.read_csv(os.path.join(DATA_DIR, 'med.csv'))
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'))

In [20]:
med_df = med_df.merge(demo_df[['stay_id', 'intime']], on='stay_id', how='left')
num_df = num_df.merge(demo_df[['stay_id', 'intime']], on='stay_id', how='left')

In [ ]:
num_df['intime'] = pd.to_datetime(num_df['intime'])
num_df['Time'] = pd.to_datetime(num_df['Time'])
num_df['time_arrive'] = (num_df['Time'] - num_df['intime']).dt.total_seconds() / 3600
num_df = num_df.drop(columns=['intime'])

In [31]:
med_df['intime'] = pd.to_datetime(med_df['intime'])
med_df['Time'] = pd.to_datetime(med_df['Time'])
med_df['time_arrive'] = (med_df['Time'] - med_df['intime']).dt.total_seconds() / 3600
med_df = med_df.drop(columns=['intime'])

In [33]:
num_df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)
med_df.to_csv(os.path.join(DATA_DIR, 'med.csv'), index=False)

LABEL

In [ ]:
stay_df = pd.read_csv(os.path.join(DATA_DIR, 'edstays.csv'), usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'disposition'])
diagnosis_df = pd.read_csv(os.path.join(DATA_DIR, 'diagnosis.csv'))

In [ ]:
diagnosis_df = diagnosis_df.merge(stay_df, on=['subject_id', 'stay_id'], how='left')
diagnosis_df = diagnosis_df[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'seq_num', 'icd_code', 'icd_version', 'icd_title']]

In [ ]:
diagnosis_df.to_csv(os.path.join(DATA_DIR, 'label_diagnosis.csv'), index=False)

In [79]:
stay_df.to_csv(os.path.join(DATA_DIR, 'label_transfer.csv'), index=False)

In [ ]:
master_df = pd.read_csv(os.path.join(DATA_DIR, 'master_dataset.csv'))
DEMO_COLS = ['gender', 'race', 'anchor_age', 'anchor_year', 'age', 'ethnicity',]
LABELS_COLS = ['intime', 'outtime', 'outcome_icu_transfer_12h', 'intime_icu', 'time_to_icu_transfer',  'time_to_icu_transfer_hours',]

SAMPLE TIME

In [ ]:
label_df = pd.read_csv('data/transfer/label.csv')

In [ ]:
label_df = label_df.drop(columns=['Trigger_time', 'Sample_time'])

In [ ]:
label_df['intime'] = pd.to_datetime(label_df['intime'])
label_df['outtime'] = pd.to_datetime(label_df['outtime'])
label_df['intime_icu'] = pd.to_datetime(label_df['intime_icu'])

In [ ]:
label_df = label_df[(label_df['outtime'] == label_df['intime_icu']) | label_df['intime_icu'].isna()]

In [ ]:
label_df = label_df.drop(columns=['time_to_icu_transfer', 'time_to_icu_transfer_hours'])

First, filter to have one example per patient id

In [ ]:
pos_df = label_df[~label_df['intime_icu'].isna()]
neg_df = label_df[label_df['intime_icu'].isna()]

In [ ]:
pos_df = pos_df.drop_duplicates()
neg_df = neg_df.drop_duplicates()

In [ ]:
pos_df = pos_df[pos_df['intime'] == pos_df.groupby('subject_id')['intime'].transform('min')]

In [ ]:
neg_df = neg_df[~neg_df['subject_id'].isin(pos_df['subject_id'])]
neg_df = neg_df[neg_df['intime'] == neg_df.groupby('subject_id')['intime'].transform('min')]

In [ ]:
pos_df['Trigger_time'] = (pos_df['outtime'] - pos_df['intime']).dt.total_seconds() / 3600
pos_df = pos_df[pos_df['Trigger_time'] >= 0.5]

In [ ]:
deltas = pos_df['Trigger_time'].values
times = np.random.choice(deltas, size=int(len(neg_df) * 1.2), replace=True) + np.random.normal(0, 0.5, int(len(neg_df) * 1.2))

In [ ]:
neg_df.loc[:, 'Sample_time'] = [t for t in times if t >= 0.5][:len(neg_df)]

/tmp/ipykernel_1402567/1178699263.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  neg_df['Sample_time'] = [t for t in times if t >= 0.5][:len(neg_df)]


In [ ]:
df = pd.concat([pos_df, neg_df], ignore_index=True)
df['Time'] = df.apply(lambda x: x['Trigger_time'] if x['outcome_icu_transfer_12h'] else x['Sample_time'], axis=1)

In [ ]:
df.to_csv('data/transfer/label.csv', index=False)

PLOT

In [ ]:
import matplotlib.pyplot as plt

# Plot histograms normalized to fraction
ax = pos_df['Trigger_time'].plot(
    kind='hist', bins=1000, alpha=0.5, label='Trigger_time', density=True, figsize=(10, 6)
)
neg_df['Sample_time'].plot(
    kind='hist', bins=1000, alpha=0.5, label='Sample_time', density=True, ax=ax
)

# Labels and legend
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Fraction')
ax.set_title('Normalized Distribution of Trigger_time and Sample_time')
ax.legend()
ax.set_xlim(0, 100)